# 🦜 LangChain LCEL — Comprehensive Revision Guide

This notebook is a **structured revision reference** for LangChain's LCEL (LangChain Expression Language)
and the Runnable abstraction layer. It is organised by concept, not by file, so ideas build on each other coherently.

---
### 📚 Table of Contents

1. [The Runnable Contract](#1-the-runnable-contract)
2. [Creating Runnables — `RunnableLambda`](#2-creating-runnables)
3. [Sequencing — `RunnableSequence` & the `|` pipe](#3-sequencing-runnables)
4. [Running Runnables — `invoke`, `batch`, `stream`](#4-running-runnables)
5. [Parallel Execution — `RunnableParallel`](#5-parallel-runnables)
6. [Combining Runnables — Full Chains with LLMs](#6-combining-runnables)
7. [Conditional Logic — `RunnableBranch`](#7-conditional-logic)
8. [Lifecycle Listeners — `with_listeners`](#8-lifecycle-listeners)
9. [Error Handling — `on_error` listener](#9-error-handling)
10. [Runtime Configuration — `configurable_fields`](#10-runtime-configuration)
11. [Persistent Configuration — `with_config`](#11-persistent-configuration)
12. [Introspection — `get_graph`, `get_prompts`](#12-introspection)
13. [End-to-End Pattern Reference](#13-end-to-end-pattern)

---


## 0. Setup

Install dependencies and load environment variables. API keys (Groq, etc.) are read from a `.env` file via `python-dotenv`.

In [ ]:
# pip install langchain langchain-groq python-dotenv  (run once if needed)

from dotenv import load_dotenv
load_dotenv()   # reads GROQ_API_KEY (or whichever keys) from .env

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableLambda,
    RunnableSequence,
    RunnableParallel,
    RunnableBranch,
    RunnableConfig,
    ConfigurableField,
)
from langchain_core.tracers.schemas import Run

# ---------------------------------------------------------------------------
# Shared model — used throughout the notebook
# ---------------------------------------------------------------------------
model = ChatGroq(model="llama-3.3-70b-versatile")
parser = StrOutputParser()


---
## 1. The Runnable Contract

### 🧠 Intuition
Every object in LangChain that can be *run* implements the **Runnable interface**.
This gives you a single, consistent API regardless of whether the object is a prompt,
a model, a parser, a custom function, or an entire chain.

### The Three Core Methods

| Method | What it does | Returns |
|---|---|---|
| `.invoke(input)` | Run once, synchronously | single output |
| `.batch([i1, i2, ...])` | Run many inputs (parallelised internally) | list of outputs |
| `.stream(input)` | Yield output chunks as they arrive | iterator of chunks |

Each also has an async twin: `ainvoke`, `abatch`, `astream`.

### Why it matters
Because everything speaks the same interface, you can **compose** any two Runnables
with the `|` operator, creating a new Runnable — that is the core idea of LCEL.

```
Runnable A  |  Runnable B  →  Runnable(A then B)
```


---
## 2. Creating Runnables — `RunnableLambda`

### 🧠 Intuition
`RunnableLambda` wraps **any Python callable** (lambda or function) so it gains
the full Runnable interface. This is the escape hatch that lets you inject
arbitrary logic into a chain.


In [ ]:
import asyncio
from langchain_core.runnables import RunnableLambda

# ── 2a. Wrapping a lambda ────────────────────────────────────────────────────
multiply = RunnableLambda(lambda x: x * 10)
print("2a:", multiply.invoke(5))   # → 50

# ── 2b. Wrapping a named function ────────────────────────────────────────────
def reverse_string(x: str) -> str:
    """Reverses the characters in a string."""
    return x[::-1]

rev = RunnableLambda(reverse_string)
print("2b:", rev.invoke("Strawberry"))   # → "yrrebwartS"

# ── 2c. Async invocation ─────────────────────────────────────────────────────
# RunnableLambda auto-adapts sync functions for async contexts
async def demo_async():
    result = await rev.ainvoke("LangChain")
    print("2c async:", result)

asyncio.run(demo_async())


### 📝 Key Points
- `RunnableLambda(fn)` works with any `fn(input) -> output` signature.
- For **dict-input** chains (most LLM chains), your function should accept and return dicts.
- Sync functions are automatically promoted to async via a thread-pool; you can also
  supply a native `async def` function for true async execution.


---
## 3. Sequencing Runnables

### 🧠 Intuition
A sequence runs Runnables **one after another**, piping output of step N as input to step N+1.
LCEL gives you three syntactically equivalent ways to express this — prefer the `|` operator.

| Style | Code |
|---|---|
| `RunnableSequence` (explicit) | `RunnableSequence(first=r1, last=r2)` |
| `.pipe()` method | `r1.pipe(r2)` |
| `\|` operator (LCEL) ✅ | `r1 \| r2` |


In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableSequence

add5  = RunnableLambda(lambda x: x + 5)
times2 = RunnableLambda(lambda x: x * 2)

# ── 3a. Explicit RunnableSequence ────────────────────────────────────────────
seq_explicit = RunnableSequence(first=add5, last=times2)
print("3a RunnableSequence:", seq_explicit.invoke(3))  # (3+5)*2 = 16

# ── 3b. .pipe() ──────────────────────────────────────────────────────────────
seq_pipe = add5.pipe(times2)
print("3b .pipe():         ", seq_pipe.invoke(3))   # same: 16

# ── 3c. | operator (LCEL) — preferred ───────────────────────────────────────
seq_lcel = add5 | times2
print("3c LCEL |:          ", seq_lcel.invoke(3))   # same: 16

# ── 3d. Longer chains — same pattern ────────────────────────────────────────
div3 = RunnableLambda(lambda x: round(x / 3, 2))
long_chain = add5 | times2 | div3
print("3d Long chain:      ", long_chain.invoke(3))  # 16/3 ≈ 5.33


### 📝 Key Points
- Data flows **left → right** through `|`.
- Every intermediate result must be a valid **input type** for the next step.
- The result of `A | B` is itself a `Runnable`, so you can keep piping: `A | B | C | ...`


---
## 4. Running Runnables — `invoke`, `batch`, `stream`

### 🧠 Intuition
You have already built a chain — now you need to **execute** it.
Choose the execution mode based on your use-case:

| Use-case | Method |
|---|---|
| One-shot Q&A, testing | `invoke` |
| Processing many items at once | `batch` |
| Streaming tokens to a UI | `stream` |


In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile")

# ── 4a. invoke — single synchronous call ─────────────────────────────────────
response = model.invoke("Where is the Eiffel Tower?")
print("4a invoke:\n", response.content[:120], "...\n")

# ── 4b. batch — multiple inputs, returns list ────────────────────────────────
prompts = [
    "President of India",
    "President of USA",
    "President of China",
]
responses = model.batch(prompts)
for p, r in zip(prompts, responses):
    print(f"  [{p}]: {r.content[:60]}...")

# ── 4c. stream — receive tokens as they arrive ───────────────────────────────
print("\n4c stream (Quantum mechanics in ~50 words):\n")
for chunk in model.stream("Explain Quantum Mechanics in 50 words."):
    print(chunk.content, end="", flush=True)
print()   # newline after stream


### 📝 Key Points
- `batch` is **not just a loop** — LangChain can parallelise the calls internally.
- `stream` works on the whole **chain** too (`chain.stream(input)`), not just the model.
- Async variants (`ainvoke`, `abatch`, `astream`) exist for every method.


---
## 5. Parallel Runnables — `RunnableParallel`

### 🧠 Intuition
`RunnableParallel` fans out **the same input** to multiple Runnables simultaneously
and collects their outputs into a **dict**. Think of it as broadcasting input,
then merging results.

```
            ┌── Runnable A ──→ output_a ─┐
input  ──→  ├── Runnable B ──→ output_b ─┤ ──→ {"a": output_a, "b": output_b}
            └── Runnable C ──→ output_c ─┘
```


In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

upper = RunnableLambda(lambda s: str(s).upper())
lower = RunnableLambda(lambda s: str(s).lower())

# ── 5a. Dict-style construction ──────────────────────────────────────────────
parallel_dict = RunnableParallel({"uppercase": upper, "lowercase": lower})
print("5a dict-style:", parallel_dict.invoke("Wonder WomAnISh"))

# ── 5b. Keyword-style construction (cleaner) ─────────────────────────────────
parallel_kw = RunnableParallel(first=upper, last=lower)
print("5b kw-style:  ", parallel_kw.invoke("Captain AmerIcA"))

# ── 5c. Pipe the parallel output into another Runnable ───────────────────────
# The dict output of RunnableParallel becomes the input to the next step
join = RunnableLambda(lambda d: d["uppercase"] + " | " + d["lowercase"])
pipeline = RunnableParallel(upper=upper, lower=lower) | join
print("5c piped:     ", pipeline.invoke("CapTain AmeRiCa"))


### 📝 Key Points
- Both Runnables inside a `RunnableParallel` **receive the same input**.
- The output is always a **`dict`** whose keys are the names you gave.
- This dict can flow directly into the next step if that step accepts named keys
  (e.g. a `ChatPromptTemplate` with `{uppercase}` and `{lowercase}` placeholders).
- Internally, LangChain may run the branches in parallel threads.


---
## 6. Combining Runnables — Full LLM Chains

### 🧠 Intuition
The real power of LCEL emerges when you combine **Prompts**, **Models**,
**Parsers**, and custom **Lambdas** into a single coherent pipeline.
The canonical pattern is:

```
prompt | model | parser
```

You can extend this by injecting extra steps, or by fanning out with
`RunnableParallel` and feeding the result into another chain.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile")
parser = StrOutputParser()

# ── 6a. The canonical chain: prompt | model | parser ─────────────────────────
prompt = ChatPromptTemplate.from_template(
    "Write a short, concise sentence about {topic}."
)
simple_chain = prompt | model | parser
result_a = simple_chain.invoke({"topic": "Quantum Mechanics"})
print("6a canonical chain:\n", result_a, "\n")

# ── 6b. Extend the chain with a post-processing Lambda ───────────────────────
annotate = RunnableLambda(lambda text: text + "  [auto-generated]")
annotated_chain = simple_chain | annotate
print("6b annotated:\n", annotated_chain.invoke({"topic": "String Theory"}), "\n")

# ── 6c. Fan-out → secondary chain (Fact-Checking pipeline) ───────────────────
#   Step 1: generate a statement about the topic (simple_chain)
#   Step 2: RunnableParallel wraps the output in {"statement": ...}
#   Step 3: checker_chain quotes it, then judges correctness
fact_check_prompt = ChatPromptTemplate.from_messages([
    ("system", "Start by quoting the statement verbatim, then assess its correctness."),
    ("user",   "How correct is this statement? {statement}"),
])
checker_chain = fact_check_prompt | model | parser

fact_checking_pipeline = (
    RunnableParallel({"statement": simple_chain})
    | checker_chain
)
result_c = fact_checking_pipeline.invoke({"topic": "String Theory"})
print("6c fact-check pipeline:\n", result_c[:300], "...")


### 📝 Key Points
- `ChatPromptTemplate` accepts a **dict** as input and outputs a `ChatPromptValue`.
- `ChatGroq` (or any ChatModel) accepts a `ChatPromptValue` and outputs an `AIMessage`.
- `StrOutputParser` converts `AIMessage` → `str`.
- The types must **chain-match** at every `|` boundary; LCEL validates this at runtime.
- `RunnableParallel({"statement": simple_chain})` transforms  
  `{"topic": "..."}` → `{"statement": "<generated sentence>"}`.  
  That dict perfectly fills `{statement}` in `fact_check_prompt`.


---
## 7. Conditional Logic — `RunnableBranch`

### 🧠 Intuition
`RunnableBranch` lets your chain **choose a path** at runtime based on a predicate.
It evaluates conditions in order and runs the first matching branch.
The last argument is the **default** branch (no condition).

```python
RunnableBranch(
    (condition_1, runnable_if_1_true),
    (condition_2, runnable_if_2_true),
    default_runnable   # ← fallback
)
```

Think of it as a `if / elif / else` block where each arm is a Runnable.


In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnableBranch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile")
parser = StrOutputParser()

# ── 7a. Generate sentence + keywords in parallel ─────────────────────────────
sentence_prompt = ChatPromptTemplate.from_template(
    "Generate a very short, concise sentence about: {topic}"
)
keywords_prompt = ChatPromptTemplate.from_template(
    "List 3-5 comma-separated keywords for: {topic}. No extra text."
)

parallel_gen = RunnableParallel(
    sentence=(sentence_prompt | model | parser),
    keywords=(keywords_prompt | model | parser),
)

# ── 7b. Define the condition (predicate must accept the dict and return bool) ─
def is_short(data: dict) -> bool:
    return len(data.get("sentence", "")) <= 80

# ── 7c. Define the two branch chains ─────────────────────────────────────────
elaborate_prompt = ChatPromptTemplate.from_messages([
    ("system", "Elaborate on the sentence using the keywords, adding detail."),
    ("user",   "Sentence: {sentence}\nKeywords: {keywords}\nElaborate:"),
])
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarise the sentence concisely using the keywords."),
    ("user",   "Sentence: {sentence}\nKeywords: {keywords}\nSummary:"),
])

elaborate_chain = elaborate_prompt | model | parser
summarize_chain = summarize_prompt | model | parser

# ── 7d. Wire the branch ──────────────────────────────────────────────────────
branch = RunnableBranch(
    (RunnableLambda(is_short), elaborate_chain),  # short → add detail
    summarize_chain,                               # long  → compress
)

full_chain = parallel_gen | branch
result = full_chain.invoke({"topic": "Black holes"})
print("7 Branch result:\n", result[:400], "...")


### 📝 Key Points
- The **condition** must be a `Runnable` (or callable wrapped in `RunnableLambda`)
  that returns a truthy / falsy value.
- The condition receives the **same input** as the branch itself.
- Branches are checked **in order**; first `True` wins.
- The final argument (no condition) is the **default / else** branch.


---
## 8. Lifecycle Listeners — `with_listeners`

### 🧠 Intuition
Listeners are **hooks** that fire at specific points in a Runnable's lifecycle
without modifying its behaviour. They are perfect for **logging, monitoring,
tracing, and debugging** in production.

```
chain.with_listeners(
    on_start = fn(run: Run) → None,
    on_end   = fn(run: Run) → None,
    on_error = fn(run: Run) → None,
)
```

The `Run` object is a snapshot of the execution metadata at that moment.


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tracers.schemas import Run

model = ChatGroq(model="llama-3.3-70b-versatile")
fact_chain = (
    ChatPromptTemplate.from_template("Give me a very short fact about {topic}.")
    | model
    | StrOutputParser()
)

# ── 8a. on_start listener ────────────────────────────────────────────────────
def on_start(run: Run):
    """Fires right before the chain executes."""
    print(f"[START] run_name={run.name!r}  run_id={run.id}")
    print(f"        inputs={run.inputs}")
    print(f"        tags={run.tags}  metadata={run.extra.get('metadata')}")

# ── 8b. on_end listener ──────────────────────────────────────────────────────
def on_end(run: Run):
    """Fires after the chain completes successfully."""
    print(f"[END]   run_name={run.name!r}  run_id={run.id}")
    print(f"        output_type={type(run.outputs).__name__}")
    print(f"        output={run.outputs}")

chain_with_listeners = fact_chain.with_listeners(
    on_start=on_start,
    on_end=on_end,
)

result = chain_with_listeners.invoke({"topic": "Quantum Physics"})
print("\nFinal result:", result)


### 📝 Key Points
- Listeners are **side-effect only** — they cannot modify input or output.
- `Run` fields you'll use most:
  - `run.id` — unique UUID for this execution
  - `run.name` — name of the Runnable (class name by default)
  - `run.inputs` / `run.outputs` — what went in / came out
  - `run.tags` / `run.extra['metadata']` — set via `RunnableConfig`
  - `run.start_time` / `run.end_time` — wall-clock timing
  - `run.parent_run_id` — links child runs to their parent (for nested chains)
- Listeners fire **synchronously in the same thread** by default.


---
## 9. Error Handling — `on_error` Listener

### 🧠 Intuition
`on_error` hooks into failures without swallowing them — the exception still
propagates, but you get a chance to **log, alert, or record** before it does.

For **recovery** (returning a fallback value), use `.with_fallbacks([...])` instead.


In [ ]:
from langchain_core.runnables import RunnableLambda
from langchain_core.tracers.schemas import Run
from typing import Any, Dict
import json

# ── 9a. A function that raises different errors depending on input ─────────────
def failing_fn(input_dict: Dict[str, Any]) -> str:
    topic = input_dict.get("topic", "").lower()
    if "error" in topic:
        raise ValueError(f"Intentional error for topic={topic!r}")
    if "network" in topic:
        raise ConnectionError("Simulated network failure")
    if "json" in topic:
        return json.loads('{"incomplete":json')  # JSON parse error
    return f"OK: processed '{topic}'"

error_runnable = RunnableLambda(failing_fn)

# ── 9b. on_error listener ────────────────────────────────────────────────────
def on_error(run: Run):
    """Fires when the Runnable raises an exception."""
    print(f"[ERROR] run_id={run.id}  name={run.name}")
    print(f"        inputs={run.inputs}")
    print(f"        start={run.start_time}  end={run.end_time}")
    print(f"        error detail: {run.error}")

guarded = error_runnable.with_listeners(on_error=on_error)

# ── 9c. Try the three failure modes ──────────────────────────────────────────
for scenario in ["some error occurred", "network issue", "valid topic"]:
    print(f"\n--- scenario: {scenario!r} ---")
    try:
        print(guarded.invoke({"topic": scenario}))
    except Exception as e:
        print(f"Caught: {type(e).__name__}: {e}")


### 📝 Key Points
- `on_error` fires **before** the exception propagates — you still need a `try/except` at the call site.
- `run.error` is a string representation of the traceback / error message.
- For **fallback chains** (silent recovery): `chain.with_fallbacks([fallback_chain])`.


---
## 10. Runtime Configuration — `configurable_fields`

### 🧠 Intuition
`configurable_fields` exposes specific parameters of a Runnable as **named knobs**
that callers can override at invocation time via `RunnableConfig`.

This lets you ship **one chain** that different callers can tune (e.g. different
token budgets or temperatures) without rebuilding the chain.

```
model.configurable_fields(
    temperature=ConfigurableField(id="temp"),
)
# later ...
chain.invoke(input, config=RunnableConfig(configurable={"temp": 0.0}))
```


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableConfig, ConfigurableField

# ── 10a. Expose max_tokens and temperature as configurable ───────────────────
configurable_model = ChatGroq(model="llama-3.3-70b-versatile").configurable_fields(
    max_tokens=ConfigurableField(
        id="token_cap",
        name="LLM Max Tokens",
        description="Maximum number of tokens in the response",
    ),
    temperature=ConfigurableField(
        id="temp_llm",
        name="LLM Temperature",
        description="Sampling temperature (0 = deterministic, 1 = creative)",
    ),
)

chain = (
    ChatPromptTemplate.from_template("Tell me an interesting fact about {topic}.")
    | configurable_model
    | StrOutputParser()
)

# ── 10b. Default invocation (no overrides) ───────────────────────────────────
result_default = chain.invoke({"topic": "Quantum Mechanics"})
print("10b Default:\n", result_default[:200], "...\n")

# ── 10c. Override: very low token limit → short answer ───────────────────────
low_token_cfg = RunnableConfig(configurable={"token_cap": 15})
result_low = chain.invoke({"topic": "Quantum Mechanics"}, config=low_token_cfg)
print("10c token_cap=15:\n", result_low)


### 📝 Key Points
- `ConfigurableField(id=...)` — the `id` is what you pass in `configurable={id: value}`.
- Only fields explicitly declared via `configurable_fields()` can be overridden.
- Works for **any** LangChain Runnable, not just LLMs.
- Use this pattern to build **one chain, many personas** (e.g. fast vs. detailed).


---
## 11. Persistent Configuration — `with_config`

### 🧠 Intuition
`with_config(RunnableConfig(...))` bakes configuration **permanently** into a chain,
so every call to that chain instance uses those settings without the caller having
to pass them each time.

Contrast:
- `chain.invoke(input, config=cfg)` — per-call override (caller must remember)
- `chain.with_config(cfg)` — returns a **new** chain with config baked in (transparent to caller)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableConfig
from langchain_core.tracers.schemas import Run

model = ChatGroq(model="llama-3.3-70b-versatile")
base_chain = (
    ChatPromptTemplate.from_template("Tell me a fact about {topic}")
    | model
    | StrOutputParser()
)

# ── 11a. Add a listener for visibility ───────────────────────────────────────
def on_start(run: Run):
    print(f"[START] tags={run.tags}  metadata={run.extra.get('metadata')}")

chain_with_listeners = base_chain.with_listeners(on_start=on_start)

# ── 11b. Bake in persistent config ───────────────────────────────────────────
persistent_cfg = RunnableConfig(
    run_name="PersistentDemo",
    tags=["persistent", "demo"],
    metadata={"user_id": "Sagar123", "source": "notebook"},
)

persistent_chain = chain_with_listeners.with_config(persistent_cfg)

# ── 11c. All invocations now carry the config automatically ──────────────────
result1 = persistent_chain.invoke({"topic": "Quantum Physics"})
print("Result 1:", result1[:80], "...\n")

result2 = persistent_chain.invoke({"topic": "General Relativity"})
print("Result 2:", result2[:80], "...")


### 📝 Key Points
- `with_config` returns a **new** Runnable; the original is unchanged.
- Useful for service-layer wrappers where you want to stamp every call with
  tenant ID, environment tag, model version, etc.
- Config keys: `run_name`, `tags`, `metadata`, `callbacks`, `configurable`, `max_concurrency`.


---
## 12. Introspection — `get_graph` & `get_prompts`

### 🧠 Intuition
LCEL chains are **self-describing**. Two useful introspection methods:

| Method | What it returns |
|---|---|
| `chain.get_graph().print_ascii()` | ASCII art of the DAG (great for debugging) |
| `chain.get_prompts()` | List of all `PromptTemplate` objects inside the chain |


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile")
parser = StrOutputParser()

sentence_chain = (
    ChatPromptTemplate.from_template("Write a sentence about {topic}")
    | model | parser
)
keywords_chain = (
    ChatPromptTemplate.from_template("List keywords for {topic}")
    | model | parser
)
parallel = RunnableParallel(sentence=sentence_chain, keywords=keywords_chain)

# ── 12a. ASCII graph ─────────────────────────────────────────────────────────
print("12a: ASCII graph of parallel chain\n")
parallel.get_graph().print_ascii()

# ── 12b. get_prompts ─────────────────────────────────────────────────────────
fact_check_prompt = ChatPromptTemplate.from_messages([
    ("system", "Quote the statement, then assess its correctness."),
    ("user",   "How correct is: {statement}?"),
])
checker = fact_check_prompt | model | parser
full_pipeline = RunnableParallel({"statement": sentence_chain}) | checker

print("\n12b: All prompts in the full pipeline\n")
for i, p in enumerate(full_pipeline.get_prompts(), 1):
    print(f"=== Prompt {i} ===")
    p.pretty_print()
    print()


### 📝 Key Points
- `get_graph()` returns a `Graph` object; call `.print_ascii()` for a quick visual,
  or `.draw_mermaid()` for a Mermaid diagram you can paste into docs.
- `get_prompts()` is handy for auditing what prompts are active across a complex chain —
  useful before deploying to production.


---
## 13. End-to-End Pattern Reference

This section is a **quick cheat-sheet** — the canonical patterns condensed.


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# LCEL PATTERN REFERENCE
# ════════════════════════════════════════════════════════════════════════════

from langchain_core.runnables import (
    RunnableLambda, RunnableParallel, RunnableBranch, RunnableConfig, ConfigurableField
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile")
parser = StrOutputParser()

# ── Pattern 1: Simple Chain ──────────────────────────────────────────────────
simple = ChatPromptTemplate.from_template("Fact about {topic}") | model | parser

# ── Pattern 2: Custom transform in chain ─────────────────────────────────────
chain_with_lambda = simple | RunnableLambda(str.upper)

# ── Pattern 3: Fan-out → merge ───────────────────────────────────────────────
parallel = RunnableParallel(a=simple, b=simple)
merge     = RunnableLambda(lambda d: d["a"] + "\n---\n" + d["b"])
fan_chain = parallel | merge

# ── Pattern 4: Conditional routing ───────────────────────────────────────────
branch_chain = parallel | RunnableBranch(
    (RunnableLambda(lambda d: len(d["a"]) < 100), simple),   # short path
    simple,                                                   # default path
)

# ── Pattern 5: Runtime-configurable chain ────────────────────────────────────
cfg_model = model.configurable_fields(
    temperature=ConfigurableField(id="temp"),
)
cfg_chain = ChatPromptTemplate.from_template("Describe {topic}") | cfg_model | parser
# invoke: cfg_chain.invoke({"topic":"..."}, config=RunnableConfig(configurable={"temp":0.0}))

# ── Pattern 6: Persistent config baked in ────────────────────────────────────
prod_chain = simple.with_config(RunnableConfig(
    run_name="prod-chain", tags=["prod"], metadata={"version": "1.0"}
))

# ── Pattern 7: Observability ─────────────────────────────────────────────────
from langchain_core.tracers.schemas import Run
observed = simple.with_listeners(
    on_start=lambda r: print(f"[START] {r.name}"),
    on_end=lambda r:   print(f"[END]   {r.name}"),
    on_error=lambda r: print(f"[ERR]   {r.error}"),
)

print("Pattern reference compiled ✓  — uncomment invocations above to test each pattern.")


---
## 🗺️ Mental Model Summary

```
LCEL Composability
══════════════════

Any two Runnables → compose with |
   A | B | C    (left-to-right data flow)

Broadcast input to N branches → RunnableParallel
   {k1: R1, k2: R2}  →  {k1: r1_out, k2: r2_out}

Route at runtime → RunnableBranch
   (condition, branch_if_true), ..., default_branch

Wrap any function → RunnableLambda(fn)

Observe without mutating → .with_listeners(on_start, on_end, on_error)

Configure at call time → .configurable_fields() + RunnableConfig(configurable={...})

Bake config permanently → .with_config(RunnableConfig(...))

Inspect structure → .get_graph().print_ascii()  /  .get_prompts()
```

---
*Notebook generated for LangChain LCEL revision — Sagar, IISER Berhampur*
